# Quantitative Trading Strategy Research Platform

**Moving Average Crossover Strategy — Full Backtest & Risk Analysis**

This notebook downloads **real historical price data** from Yahoo Finance,
implements a systematic Moving Average Crossover strategy, backtests it
against a Buy & Hold baseline and the S&P 500 (SPY), computes institutional
risk metrics, and visualizes the results.

No synthetic or placeholder data is used — every number below comes from an
actual `yfinance` download at run time.

Run cells top to bottom. Change `TICKER`, `START_DATE`, `END_DATE` in the
config cell to test other symbols/windows.


In [ ]:
# ============================================================
# SETUP — install & import dependencies
# ============================================================
# --upgrade matters: Colab often ships an older cached yfinance, and Yahoo
# Finance changes its response format frequently enough that an old
# yfinance version is the #1 cause of "it worked before, now it's broken".
!pip install -q --upgrade yfinance plotly

import time
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import yfinance as yf

# Force the Colab-compatible renderer explicitly. Plotly's auto-detected
# default renderer is not always "colab" (depends on how the runtime was
# started), which is the most common reason charts silently don't appear.
pio.renderers.default = "colab"

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
np.random.seed(42)

print("yfinance version:", yf.__version__)


> **If you still see import errors after this cell:** go to
> `Runtime > Restart session`, then `Runtime > Run all`. Colab sometimes
> keeps an old yfinance build loaded in memory even after `pip install
> --upgrade` finishes — a restart forces the new version to load.


## 1. Configuration

In [ ]:
# ============================================================
# CONFIG — change these to test a different ticker / window
# ============================================================
TICKER = "AAPL"          # stock to trade
BENCHMARK = "SPY"        # benchmark for beta/alpha and comparison
START_DATE = "2019-01-01"
END_DATE = "2024-12-31"
STARTING_CAPITAL = 10_000.0

SHORT_WINDOW = 20        # short-term moving average (days)
LONG_WINDOW = 50         # long-term moving average (days)

RISK_FREE_RATE = 0.02    # annualized, used in Sharpe/Sortino
TRADING_DAYS_PER_YEAR = 252


## 2. Data Pipeline

Downloads historical OHLCV data for both the traded ticker and the
benchmark, validates it, and computes daily returns. `yfinance` returns
multi-level columns for single-ticker downloads in current versions, so we
flatten them before use.


In [ ]:
def _flatten_columns(df):
    """Flatten yfinance's column index to simple field names.

    Different yfinance versions return single-ticker downloads with either
    plain columns (Open, High, Low, Close, Volume) or MultiIndex columns —
    and the MultiIndex level order (field-then-ticker vs ticker-then-field)
    has also changed across versions. This checks which level actually
    contains recognizable field names instead of assuming a fixed order.
    """
    if not isinstance(df.columns, pd.MultiIndex):
        return df

    field_names = {"Open", "High", "Low", "Close", "Adj Close", "Volume"}
    level0 = set(df.columns.get_level_values(0))
    if field_names & level0:
        df.columns = df.columns.get_level_values(0)
    else:
        df.columns = df.columns.get_level_values(1)
    return df


def download_price_data(ticker, start, end, max_retries=3, retry_delay=2):
    """Download and clean historical price data for a single ticker.

    Retries transient failures (rate limiting, dropped connections) and
    raises a clear, actionable error if data genuinely can't be retrieved,
    instead of letting a cryptic pandas/requests traceback crash the rest
    of the notebook.
    """
    last_error = None
    df = None

    for attempt in range(1, max_retries + 1):
        try:
            df = yf.download(
                ticker, start=start, end=end,
                auto_adjust=True, progress=False, threads=False,
            )
            if df is not None and not df.empty:
                break
            last_error = "Yahoo Finance returned an empty dataset."
        except Exception as e:
            last_error = e

        if attempt < max_retries:
            print(f"  [{ticker}] attempt {attempt} failed ({last_error}); "
                  f"retrying in {retry_delay}s...")
            time.sleep(retry_delay)

    if df is None or df.empty:
        raise RuntimeError(
            f"Could not download data for '{ticker}' after {max_retries} attempts.\n"
            f"Last error: {last_error}\n\n"
            "Common causes:\n"
            "  - Ticker symbol is misspelled or delisted\n"
            "  - START_DATE/END_DATE window has no trading days for this symbol\n"
            "  - Yahoo Finance is temporarily rate-limiting this session — wait "
            "a minute and re-run this cell\n"
            "  - Colab's yfinance version is stale — Runtime > Restart session, "
            "then Runtime > Run all"
        )

    df = _flatten_columns(df)

    if "Close" not in df.columns:
        raise RuntimeError(
            f"Downloaded data for '{ticker}' is missing a 'Close' column "
            f"(columns found: {list(df.columns)}). This usually means "
            "Yahoo Finance's response format changed — try upgrading "
            "yfinance again (`pip install --upgrade yfinance`)."
        )

    # --- validation & cleaning ---
    missing_before = int(df["Close"].isna().sum())
    df["Close"] = df["Close"].ffill()  # forward-fill isolated gaps
    df = df.dropna(subset=["Close"])   # drop any remaining unrecoverable gaps
    missing_after = int(df["Close"].isna().sum())

    df["daily_return"] = df["Close"].pct_change()
    df = df.dropna(subset=["daily_return"])

    if len(df) < 60:
        raise RuntimeError(
            f"Only {len(df)} usable trading days for '{ticker}' after cleaning — "
            f"too few to backtest a {SHORT_WINDOW}/{LONG_WINDOW}-day moving "
            "average strategy. Widen START_DATE/END_DATE."
        )

    print(f"{ticker}: {len(df)} trading days downloaded "
          f"({df.index.min().date()} to {df.index.max().date()})")
    print(f"{ticker}: filled {missing_before} missing values, {missing_after} remain")

    return df

try:
    price_data = download_price_data(TICKER, START_DATE, END_DATE)
    benchmark_data = download_price_data(BENCHMARK, START_DATE, END_DATE)
except RuntimeError as e:
    print("DATA PIPELINE ERROR:\n")
    print(e)
    raise

price_data.tail()


## 3. Strategy: Moving Average Crossover

**Logic:** go long when the short-term moving average is above the
long-term moving average (an uptrend signal), stay flat otherwise.

**Signal is shifted forward one day** before being applied to returns — a
trade decided on today's close can only be acted on starting the next
trading day. Skipping this step is the single most common bug in beginner
backtests (it silently trades on information not yet available), so it's
called out explicitly here.


In [ ]:
def sma_crossover_signals(df, short_window, long_window):
    """Generate long/flat signals from a moving average crossover."""
    out = df.copy()
    out["sma_short"] = out["Close"].rolling(short_window).mean()
    out["sma_long"] = out["Close"].rolling(long_window).mean()

    out["signal"] = 0
    out.loc[out["sma_short"] > out["sma_long"], "signal"] = 1

    # Shift by 1 day: trade on tomorrow's open based on yesterday's close crossover
    out["signal"] = out["signal"].shift(1).fillna(0)

    return out.dropna(subset=["sma_long"])  # drop initial warm-up period

strategy_df = sma_crossover_signals(price_data, SHORT_WINDOW, LONG_WINDOW)
print("Days in market (long):", int(strategy_df["signal"].sum()))
print("Days flat:", int((strategy_df["signal"] == 0).sum()))
strategy_df[["Close", "sma_short", "sma_long", "signal"]].tail()


**Limitations of this strategy:** moving average crossovers lag price
(they only confirm a trend after it has partly happened), tend to whipsaw
in sideways/choppy markets generating repeated small losses, and use no
volatility or volume filter. This implementation also ignores transaction
costs, bid-ask spread, and slippage — all of which would reduce real-world
returns relative to what's shown below.


## 4. Backtesting Engine

Simulates the strategy against **$10,000 starting capital**: applies the
signal to daily returns to build an equity curve, and reconstructs a trade
log from every long-entry/long-exit pair for trade-level statistics.


In [ ]:
def run_backtest(df, starting_capital):
    """Simulate portfolio value under the strategy vs. buy & hold."""
    bt = df.copy()
    bt["strategy_return"] = bt["signal"] * bt["daily_return"]

    bt["equity_curve"] = starting_capital * (1 + bt["strategy_return"]).cumprod()
    bt["buy_hold_equity"] = starting_capital * (1 + bt["daily_return"]).cumprod()

    # --- trade log: pair each entry (0->1) with its exit (1->0) ---
    signal_change = bt["signal"].diff().fillna(0)
    entries = bt.index[signal_change == 1]
    exits = bt.index[signal_change == -1]

    trade_records = []
    for entry_date in entries:
        later_exits = exits[exits > entry_date]
        exit_date = later_exits[0] if len(later_exits) > 0 else bt.index[-1]
        entry_price = bt.loc[entry_date, "Close"]
        exit_price = bt.loc[exit_date, "Close"]
        trade_records.append({
            "entry_date": entry_date,
            "exit_date": exit_date,
            "entry_price": round(float(entry_price), 2),
            "exit_price": round(float(exit_price), 2),
            "pnl_pct": float(exit_price / entry_price - 1),
            "holding_days": (exit_date - entry_date).days,
        })

    trade_log = pd.DataFrame(trade_records)
    return bt, trade_log

backtest_df, trade_log = run_backtest(strategy_df, STARTING_CAPITAL)

n_trades = len(trade_log)
winners = int((trade_log["pnl_pct"] > 0).sum()) if n_trades else 0
losers = int((trade_log["pnl_pct"] <= 0).sum()) if n_trades else 0
win_rate = winners / n_trades if n_trades else 0.0

final_strategy_value = backtest_df["equity_curve"].iloc[-1]
final_buyhold_value = backtest_df["buy_hold_equity"].iloc[-1]

print(f"Starting capital:        ${STARTING_CAPITAL:,.2f}")
print(f"Final strategy value:    ${final_strategy_value:,.2f}")
print(f"Final buy & hold value:  ${final_buyhold_value:,.2f}")
print(f"Total trades:            {n_trades}")
print(f"Winning trades:          {winners}")
print(f"Losing trades:           {losers}")
print(f"Win rate:                {win_rate:.1%}")

trade_log.head(10)


## 5. Risk & Performance Metrics

Standard institutional performance metrics, computed from the daily
strategy return series:

- **Total Return** — cumulative percentage gain/loss over the full period
- **Annualized Return** — total return rescaled to a 1-year equivalent (CAGR-style)
- **Annualized Volatility** — standard deviation of daily returns, annualized — the
  standard measure of risk
- **Sharpe Ratio** — excess return per unit of total risk: `(annualized return − risk-free rate) / volatility`
- **Sortino Ratio** — like Sharpe, but only penalizes *downside* volatility, since
  investors don't mind upside swings
- **Maximum Drawdown** — the worst peak-to-trough decline in portfolio value —
  a direct measure of "how bad could it have gotten"
- **Beta** — sensitivity to the benchmark's moves (market risk exposure)
- **Alpha** — annualized return in excess of what beta-adjusted market
  exposure alone would predict (CAPM-style) — the actual value the
  strategy added


In [ ]:
def compute_risk_metrics(strategy_returns, benchmark_returns,
                          risk_free_rate=0.02, periods_per_year=252):
    """Compute institutional risk & performance metrics for a return series."""
    strategy_returns = strategy_returns.dropna()
    benchmark_returns = benchmark_returns.reindex(strategy_returns.index).dropna()
    strategy_returns = strategy_returns.reindex(benchmark_returns.index).dropna()

    n_days = len(strategy_returns)
    total_return = (1 + strategy_returns).prod() - 1
    annualized_return = (1 + total_return) ** (periods_per_year / n_days) - 1
    annualized_vol = strategy_returns.std() * np.sqrt(periods_per_year)

    sharpe = ((annualized_return - risk_free_rate) / annualized_vol
              if annualized_vol != 0 else np.nan)

    downside_returns = strategy_returns[strategy_returns < 0]
    downside_vol = downside_returns.std() * np.sqrt(periods_per_year)
    sortino = ((annualized_return - risk_free_rate) / downside_vol
               if downside_vol and downside_vol != 0 else np.nan)

    equity = (1 + strategy_returns).cumprod()
    running_max = equity.cummax()
    drawdown_series = (equity - running_max) / running_max
    max_drawdown = drawdown_series.min()

    covariance = np.cov(strategy_returns, benchmark_returns)
    beta = covariance[0, 1] / covariance[1, 1] if covariance[1, 1] != 0 else np.nan

    bench_total_return = (1 + benchmark_returns).prod() - 1
    bench_annualized_return = (1 + bench_total_return) ** (periods_per_year / len(benchmark_returns)) - 1
    alpha = annualized_return - (risk_free_rate + beta * (bench_annualized_return - risk_free_rate))

    return {
        "Total Return": total_return,
        "Annualized Return": annualized_return,
        "Annualized Volatility": annualized_vol,
        "Sharpe Ratio": sharpe,
        "Sortino Ratio": sortino,
        "Maximum Drawdown": max_drawdown,
        "Beta": beta,
        "Alpha": alpha,
    }, drawdown_series

strategy_metrics, drawdown_series = compute_risk_metrics(
    backtest_df["strategy_return"], benchmark_data["daily_return"],
    RISK_FREE_RATE, TRADING_DAYS_PER_YEAR
)
buyhold_metrics, _ = compute_risk_metrics(
    backtest_df["daily_return"], benchmark_data["daily_return"],
    RISK_FREE_RATE, TRADING_DAYS_PER_YEAR
)
benchmark_metrics, _ = compute_risk_metrics(
    benchmark_data["daily_return"], benchmark_data["daily_return"],
    RISK_FREE_RATE, TRADING_DAYS_PER_YEAR
)

metrics_table = pd.DataFrame({
    f"{TICKER} — SMA Strategy": strategy_metrics,
    f"{TICKER} — Buy & Hold": buyhold_metrics,
    f"{BENCHMARK} — Benchmark": benchmark_metrics,
})

pct_rows = ["Total Return", "Annualized Return", "Annualized Volatility", "Maximum Drawdown", "Alpha"]
ratio_rows = ["Sharpe Ratio", "Sortino Ratio", "Beta"]

formatted_table = metrics_table.astype(object).copy()
for row in pct_rows:
    formatted_table.loc[row] = metrics_table.loc[row].apply(lambda x: f"{x:.2%}")
for row in ratio_rows:
    formatted_table.loc[row] = metrics_table.loc[row].apply(lambda x: f"{x:.3f}")

formatted_table


## 6. Benchmarking Summary

In [ ]:
print(f"=== {TICKER} SMA({SHORT_WINDOW}/{LONG_WINDOW}) Strategy vs. Benchmarks ===\n")
print(f"Strategy total return:    {strategy_metrics['Total Return']:.2%}")
print(f"Buy & Hold total return:  {buyhold_metrics['Total Return']:.2%}")
print(f"{BENCHMARK} total return:{' ' * (14 - len(BENCHMARK))}{benchmark_metrics['Total Return']:.2%}")
print()
print(f"Strategy Sharpe ratio:    {strategy_metrics['Sharpe Ratio']:.3f}")
print(f"Buy & Hold Sharpe ratio:  {buyhold_metrics['Sharpe Ratio']:.3f}")
print()
outperformed = strategy_metrics["Total Return"] > buyhold_metrics["Total Return"]
print(f"Strategy {'OUTPERFORMED' if outperformed else 'UNDERPERFORMED'} buy & hold "
      f"on this ticker over this exact window.")
print("This is one backtest over one historical window — not a general claim "
      "that the strategy 'works'. Results are highly sensitive to the ticker, "
      "date range, and parameter choices tested.")


## 7. Visualization

All charts below are built with Plotly from the actual backtest output —
nothing here is a mock or sample chart.


In [ ]:
# --- 7.1 Equity Curve: Strategy vs Buy & Hold ---
fig_equity = go.Figure()
fig_equity.add_trace(go.Scatter(
    x=backtest_df.index, y=backtest_df["equity_curve"],
    name=f"{TICKER} SMA Strategy", line=dict(color="#2E86AB", width=2)
))
fig_equity.add_trace(go.Scatter(
    x=backtest_df.index, y=backtest_df["buy_hold_equity"],
    name=f"{TICKER} Buy & Hold", line=dict(color="#A23B72", width=2, dash="dash")
))
fig_equity.update_layout(
    title=f"Portfolio Equity Curve — {TICKER} (Starting Capital ${STARTING_CAPITAL:,.0f})",
    xaxis_title="Date", yaxis_title="Portfolio Value ($)",
    template="plotly_white", hovermode="x unified"
)
fig_equity.show()


In [ ]:
# --- 7.2 Drawdown Chart ---
fig_dd = go.Figure()
fig_dd.add_trace(go.Scatter(
    x=drawdown_series.index, y=drawdown_series.values * 100,
    fill="tozeroy", name="Drawdown", line=dict(color="#C73E1D")
))
fig_dd.update_layout(
    title=f"Strategy Drawdown — {TICKER}",
    xaxis_title="Date", yaxis_title="Drawdown (%)",
    template="plotly_white"
)
fig_dd.show()


In [ ]:
# --- 7.3 Monthly Returns Heatmap ---
monthly_returns = (
    backtest_df["strategy_return"]
    .add(1)
    .resample("ME")
    .prod()
    .sub(1)
)
heatmap_df = monthly_returns.to_frame("return")
heatmap_df["year"] = heatmap_df.index.year
heatmap_df["month"] = heatmap_df.index.strftime("%b")
month_order = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
pivot = heatmap_df.pivot(index="year", columns="month", values="return").reindex(columns=month_order)

fig_heat = go.Figure(data=go.Heatmap(
    z=(pivot.values * 100), x=pivot.columns, y=pivot.index.astype(str),
    colorscale="RdYlGn", zmid=0,
    text=[[f"{v:.1f}%" if not np.isnan(v) else "" for v in row] for row in (pivot.values * 100)],
    texttemplate="%{text}", colorbar=dict(title="Return %")
))
fig_heat.update_layout(
    title=f"Monthly Returns Heatmap — {TICKER} Strategy",
    xaxis_title="Month", yaxis_title="Year", template="plotly_white"
)
fig_heat.show()


In [ ]:
# --- 7.4 Trade Entry/Exit Markers on Price Chart ---
fig_trades = go.Figure()
fig_trades.add_trace(go.Scatter(
    x=backtest_df.index, y=backtest_df["Close"],
    name=f"{TICKER} Close", line=dict(color="#444444", width=1)
))
fig_trades.add_trace(go.Scatter(
    x=backtest_df.index, y=backtest_df["sma_short"],
    name=f"SMA {SHORT_WINDOW}", line=dict(color="#2E86AB", width=1, dash="dot")
))
fig_trades.add_trace(go.Scatter(
    x=backtest_df.index, y=backtest_df["sma_long"],
    name=f"SMA {LONG_WINDOW}", line=dict(color="#F18F01", width=1, dash="dot")
))
if not trade_log.empty:
    fig_trades.add_trace(go.Scatter(
        x=trade_log["entry_date"], y=trade_log["entry_price"],
        mode="markers", name="Entry", marker=dict(symbol="triangle-up", size=11, color="green")
    ))
    fig_trades.add_trace(go.Scatter(
        x=trade_log["exit_date"], y=trade_log["exit_price"],
        mode="markers", name="Exit", marker=dict(symbol="triangle-down", size=11, color="red")
    ))
fig_trades.update_layout(
    title=f"{TICKER} Price with SMA Crossover Trade Signals",
    xaxis_title="Date", yaxis_title="Price ($)", template="plotly_white", hovermode="x unified"
)
fig_trades.show()


In [ ]:
# --- 7.5 Risk-Return Comparison ---
comparison = pd.DataFrame({
    "Strategy": [f"{TICKER} SMA Strategy", f"{TICKER} Buy & Hold", f"{BENCHMARK} Benchmark"],
    "Annualized Return": [strategy_metrics["Annualized Return"],
                           buyhold_metrics["Annualized Return"],
                           benchmark_metrics["Annualized Return"]],
    "Annualized Volatility": [strategy_metrics["Annualized Volatility"],
                              buyhold_metrics["Annualized Volatility"],
                              benchmark_metrics["Annualized Volatility"]],
    "Sharpe Ratio": [strategy_metrics["Sharpe Ratio"],
                     buyhold_metrics["Sharpe Ratio"],
                     benchmark_metrics["Sharpe Ratio"]],
})

fig_rr = go.Figure(data=go.Scatter(
    x=comparison["Annualized Volatility"] * 100,
    y=comparison["Annualized Return"] * 100,
    mode="markers+text", text=comparison["Strategy"], textposition="top center",
    marker=dict(size=comparison["Sharpe Ratio"].fillna(0).abs() * 15 + 10,
                color=["#2E86AB", "#A23B72", "#F18F01"])
))
fig_rr.update_layout(
    title="Risk-Return Comparison (marker size ~ |Sharpe Ratio|)",
    xaxis_title="Annualized Volatility (%)", yaxis_title="Annualized Return (%)",
    template="plotly_white"
)
fig_rr.show()


In [ ]:
# --- 7.6 Portfolio Allocation (time in market vs. cash) ---
days_long = int(backtest_df["signal"].sum())
days_flat = int((backtest_df["signal"] == 0).sum())

fig_alloc = go.Figure(data=go.Pie(
    labels=[f"Invested in {TICKER}", "In Cash (flat)"],
    values=[days_long, days_flat],
    marker=dict(colors=["#2E86AB", "#CCCCCC"]), hole=0.4
))
fig_alloc.update_layout(
    title="Strategy Time Allocation: Invested vs. Cash",
    template="plotly_white"
)
fig_alloc.show()


## Limitations

- No transaction costs, slippage, or bid-ask spread modeled — real returns would be lower.
- Single-asset, long-only strategy — no shorting, no diversification.
- One historical window on one ticker is not statistically sufficient to
  conclude a strategy "works" — proper validation needs out-of-sample testing,
  multiple tickers, and multiple market regimes.
- Past performance shown here is historical simulation, not a forecast.

## Next Steps

- Parametrize and grid-search `SHORT_WINDOW` / `LONG_WINDOW` to check
  sensitivity (careful: this risks overfitting if not validated out-of-sample).
- Add the Momentum and Mean Reversion strategies from the platform's
  `strategies/` module and compare all three head-to-head.
- Add transaction cost and slippage assumptions to the backtest engine.
